# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [2]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 402.41 GB
MemAvailable: 874.23 GB
Free GPU Memory (GB): 39.3896

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.c

## 2. SEML Pipeline

In [4]:
exp_id = "09-01-1"

In [5]:
from src.data.FKTC_datasets import load_dataset_from_name
from src.reliability.response_generator import ResponseGenerator
from src.models import base_models
import torch

import pandas as pd
import numpy as np
from sklearn import metrics

def run_reliability_eval(
    model_name,
    max_new_tokens,
    temperature,
    use_beam_search,
    strategy,
    dataset_name,
    taxonomy_type,
    device='cuda',
    seed=123,
    n_repeats=5,
    n_beams=5,
    cache_path=CACHE_PATH
    ):
    # LOAD DATASET
    qa_dataset = load_dataset_from_name(dataset_name, max_entries=10, taxonomy_type=taxonomy_type)

    # INITIALIZE RESULTS LIST
    results = []

    n_steps = 0
    total_steps = len(qa_dataset) * (n_repeats if not use_beam_search else 1)
    generator = ResponseGenerator(base_models[model_name])
    for query_idx, (query, true_answer) in enumerate(qa_dataset):
        run_results = generator.generate_response(query, strategy, true_answer, max_new_tokens, temperature, use_beam_search, n_repeats=n_repeats, n_beams=n_beams)
        for result_dict in run_results:
            print(f"  TOTAL: {n_steps + 1}/{total_steps}, MODEL: {model_name}, QUERY: {query_idx}, STRATEGY: {strategy}, MAX_NEW_TOKENS: {max_new_tokens}, RUN: {result_dict['run']}/{n_repeats}")
            # Store the results in the list
            results.append({
                "Query ID": query_idx,
                "Query": query,
                "Answer": true_answer,
                "Run": result_dict['run'],
                "Generated Response": result_dict['output_text'],
                "Cleaned": result_dict['cleaned'],
                "P": result_dict['beam_prob'],
                "P_adj": result_dict['beam_prob_adj'],
                "Entropy": result_dict['entropy'],
                "Is Correct": result_dict['is_correct'],
                "Token Probabilities": result_dict['token_probs']
            })
            print(f"    IS_CORRECT: {result_dict['is_correct']}, CLEANED: {result_dict['cleaned']}, PROB: {result_dict['beam_prob']:.2f}, ADJ_PROB: {result_dict['beam_prob_adj']:.2f}, ENTROPY: {result_dict['entropy']:.2f}")
            n_steps += 1
    
    # Generate custom file name based on parameters
    beam_search_str = "beam" if use_beam_search else "sample"
    strategy_str = strategy.replace(" ", "_").lower()  # Replace spaces with underscores for file names
    file_base = f"{model_name}_{dataset_name}_{taxonomy_type}_{beam_search_str}_{max_new_tokens}_tokens_{temperature}_temp_{strategy_str}"

    # Optional: Create a directory for saving the results if not already existing
    results_path = "/nfs/homedirs/daro/git/quantization-reliability/results"
    save_dir = os.path.join(results_path, "reliability_eval")
    os.makedirs(save_dir, exist_ok=True)

    # Generate file paths
    exp_path = os.path.join(save_dir, f"reliability_eval_{exp_id}")
    os.makedirs(exp_path, exist_ok=True)
    
    raw_table_path = os.path.join(exp_path, f"{file_base}_raw_table_{exp_id}.xlsx")
    scores_table_path = os.path.join(exp_path, f"{file_base}_scores_{exp_id}.xlsx")
    
    df_results = pd.DataFrame(results)
    
    # Calculate P_sem as the proportion of True values in 'Is Correct' per group
    df_results['P_sem'] = df_results.groupby(['Query ID'])['Is Correct'].transform('mean')

    # Define custom AUC calculation
    def custom_auc_roc(corrects, scores):
        fpr, tpr, thresholds = metrics.roc_curve(corrects, scores)
        return metrics.auc(fpr, tpr)
    
    def calculate_scores(df):
        y_true = df['Is Correct'].values

        # Calculate various AUCROC and AUCPR scores
        scores_dict = {}
        metrics_to_calculate = {
            'sample': 'P',
            'adj': 'P_adj',
            'entr': 'Entropy',
            'sem': 'P_sem'
        }

        for key, score_column in metrics_to_calculate.items():
            y_scores = df[score_column].values
            if len(set(y_true)) > 1:  # Ensure at least two classes are present
                aucroc = custom_auc_roc(y_true, y_scores)
            else:
                aucroc = np.nan
            aucpr = metrics.average_precision_score(y_true, y_scores)
            accuracy = np.mean(y_true)

            scores_dict[f'AUCROC_{key}'] = aucroc
            scores_dict[f'AUCPR_{key}'] = aucpr
        
        scores_dict['Accuracy'] = accuracy
    
        return pd.Series(scores_dict)

    # Apply the calculate_scores function to the entire DataFrame
    df_scores = calculate_scores(df_results)

    # Convert df_scores to a DataFrame with a single row for consistent saving format
    df_scores = df_scores.to_frame().T

    # Save the original detailed results to an Excel file
    df_results.to_excel(raw_table_path, index=False)
    df_scores.to_excel(scores_table_path, index=False)

    return {"results": df_results, "scores": df_scores}

In [6]:
import itertools

# Fixed parameters
fixed_params = {
    'device': 'cuda',
    'seed': 123,
    'n_repeats': 3,
    'n_beams': 5,
    'cache_path': CACHE_PATH
}

# Grid parameters
grid_params = {
    'model_name': [
        'Llama-3-8B',
        # 'TinyLlama-Chat',
        # 'Bloomz',
        # 'GPT2-Large',
        # 'TinyLlama',
        # 'Llama-3-8B-AWQ-4bit'  # Uncomment if needed
    ],
    'max_new_tokens': [
        # 10,
        # 15,
        20,
        # 30,
        # 40
    ],
    'temperature': [
        0.1
    ],
    'use_beam_search': [
        # True,
        False
    ],
    'strategy': [
        "Fact Statement",
        "Completion",
        "Definitive Statement",
        "Fill-in-the-Blank",
        "Structured Answer Prompt",
        "Direct Instruction",
        # "Contextual Prompts",
        # "Question-Answer Pairs",
        # "Direct Answer",
        # "Q&A Format",
        # "Instructional",
        # "Summary",
        # "True Completion",
        # "Direct Completion",
        # "Answer Completion",
        # "Direct Query",
        # "Factual Retrieval",
        # "First Thought",
        # "Deductive Reasoning"
    ],
    'dataset_name': [
        # 'toy-qa-dataset',
        'P101',
        # 'P103',
        # 'P108',
        # 'P127',
        # 'P1376',
        # 'P1412',
        # 'P159',
        # 'P17',
        # 'P176',
        # 'P178',
        # 'P19',
        # 'P20',
        # 'P264',
        # 'P27',
        # 'P276',
        # 'P30',
        # 'P364',
        # 'P37',
        # 'P495',
        # 'P740'
    ],
    "taxonomy_type": [
        "0",
        "pos",
        "neg1",
        # "neg2",
        # "neg3",
        # "neg4",
        # "neg5"
    ]
}

# Generate all combinations for grid search
grid_combinations = list(itertools.product(
    grid_params['model_name'],
    grid_params['max_new_tokens'],
    grid_params['temperature'],
    grid_params['use_beam_search'],
    grid_params['strategy'],
    grid_params['dataset_name'],
    grid_params['taxonomy_type']
))

# Run the quantize function for all combinations
results = []
max_combinations = 10000  # Set to a lower number for testing purposes
for i, combination in enumerate(grid_combinations):
    if i >= max_combinations:
        break
    model_name, max_new_tokens, temperature, use_beam_search, strategy, dataset_name, taxonomy_type = combination

    # Print current combination details
    print(f"Running combination {i+1}/{len(grid_combinations)}")
    print(f"  Model Name: {model_name}")
    print(f"  Max New Tokens: {max_new_tokens}")
    print(f"  Temperature: {temperature}")
    print(f"  Use Beam Search: {use_beam_search}")
    print(f"  Strategy: {strategy}")
    print(f"  Dataset Name: {dataset_name}")
    print(f"  Taxonomy Type: {taxonomy_type}")

    result = run_reliability_eval(
        model_name=model_name,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        use_beam_search=use_beam_search,
        strategy=strategy,
        dataset_name=dataset_name,
        taxonomy_type=taxonomy_type,
        device=fixed_params['device'],
        seed=fixed_params['seed'],
        n_repeats=fixed_params['n_repeats'],
        n_beams=fixed_params['n_beams'],
        cache_path=CACHE_PATH
    )

    # Append result with parameter details
    results.append({
        'result': result,
        'parameters': {
            'model_name': model_name,
            'max_new_tokens': max_new_tokens,
            'temperature': temperature,
            'use_beam_search': use_beam_search,
            'strategy': strategy,
            'dataset_name': dataset_name,
            'taxonomy_type': taxonomy_type,
            'device': fixed_params['device'],
            'n_repeats': fixed_params['n_repeats'],
            'n_beams': fixed_params['n_beams']
        }
    })

# Do something with the results
print(results)

Running combination 1/18
  Model Name: Llama-3-8B
  Max New Tokens: 20
  Temperature: 0.1
  Use Beam Search: False
  Strategy: Fact Statement
  Dataset Name: P101
  Taxonomy Type: 0


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:13<00:00,  3.42s/it]


  TOTAL: 1/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 1/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.50, ADJ_PROB: 0.97, ENTROPY: 0.35
  TOTAL: 2/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 2/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.16, ADJ_PROB: 0.91, ENTROPY: 0.97
  TOTAL: 3/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 3/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.50, ADJ_PROB: 0.97, ENTROPY: 0.35
  TOTAL: 4/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 1/3
    IS_CORRECT: False, CLEANED: True, PROB: 1.00, ADJ_PROB: 1.00, ENTROPY: -0.00
  TOTAL: 5/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 2/3
    IS_CORRECT: False, CLEANED: True, PROB: 1.00, ADJ_PROB: 1.00, ENTROPY: -0.00
  TOTAL: 6/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 3/3
    

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.08s/it]


  TOTAL: 1/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 1/3
    IS_CORRECT: False, CLEANED: True, PROB: 1.00, ADJ_PROB: 1.00, ENTROPY: -0.00
  TOTAL: 2/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 2/3
    IS_CORRECT: False, CLEANED: True, PROB: 1.00, ADJ_PROB: 1.00, ENTROPY: -0.00
  TOTAL: 3/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 3/3
    IS_CORRECT: False, CLEANED: True, PROB: 1.00, ADJ_PROB: 1.00, ENTROPY: -0.00
  TOTAL: 4/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 1/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.50, ADJ_PROB: 0.97, ENTROPY: 0.35
  TOTAL: 5/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 2/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.50, ADJ_PROB: 0.97, ENTROPY: 0.35
  TOTAL: 6/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 20, RUN: 3/3
   

...

  TOTAL: 1/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 1/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.78, ADJ_PROB: 0.99, ENTROPY: 0.20
  TOTAL: 2/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 2/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.11, ADJ_PROB: 0.90, ENTROPY: 0.68
  TOTAL: 3/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 3/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.78, ADJ_PROB: 0.99, ENTROPY: 0.20
  TOTAL: 4/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 1/3
    IS_CORRECT: False, CLEANED: True, PROB: 1.00, ADJ_PROB: 1.00, ENTROPY: -0.00
  TOTAL: 5/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 2/3
    IS_CORRECT: False, CLEANED: True, PROB: 1.00, ADJ_PROB: 1.00, ENTROPY: -0.00
  TOTAL: 6/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Direct Instruction, MAX_NEW_T

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.09s/it]


  TOTAL: 1/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 1/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.11, ADJ_PROB: 0.90, ENTROPY: 0.99
  TOTAL: 2/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 2/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.33, ADJ_PROB: 0.95, ENTROPY: 0.63
  TOTAL: 3/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 3/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.33, ADJ_PROB: 0.95, ENTROPY: 0.63
  TOTAL: 4/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 1/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.07, ADJ_PROB: 0.88, ENTROPY: 0.96
  TOTAL: 5/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 2/3
    IS_CORRECT: True, CLEANED: True, PROB: 0.50, ADJ_PROB: 0.97, ENTROPY: 0.35
  TOTAL: 6/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Direct Instruction, MAX_NEW_TOKE

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.11s/it]


  TOTAL: 1/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 1/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.35, ADJ_PROB: 0.95, ENTROPY: 0.37
  TOTAL: 2/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 2/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.65, ADJ_PROB: 0.98, ENTROPY: 0.28
  TOTAL: 3/30, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 3/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.35, ADJ_PROB: 0.95, ENTROPY: 0.37
  TOTAL: 4/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 1/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.78, ADJ_PROB: 0.99, ENTROPY: 0.20
  TOTAL: 5/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Direct Instruction, MAX_NEW_TOKENS: 20, RUN: 2/3
    IS_CORRECT: False, CLEANED: True, PROB: 0.78, ADJ_PROB: 0.99, ENTROPY: 0.20
  TOTAL: 6/30, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Direct Instruction, MAX_NEW_TOK

## 3. Unify Excel Files

In [ ]:
import os
import pandas as pd

def unify_excel_files(save_dir):
    # Define paths for the unified raw table and scores table
    unified_raw_table_path = os.path.join(save_dir, f"unified_raw_table_{exp_id}.xlsx")
    unified_scores_table_path = os.path.join(save_dir, f"unified_scores_table_{exp_id}.xlsx")
    
    # Initialize empty DataFrames for unified tables
    unified_raw_table = pd.DataFrame()
    unified_scores_table = pd.DataFrame()
    
    exclude_files = [f"unified_raw_table_{exp_id}.xlsx", f"unified_scores_table_{exp_id}.xlsx"]

    # Loop through all files in the save_dir
    exp_dir = os.path.join(save_dir, f"reliability_eval_{exp_id}")
    for filename in os.listdir(exp_dir):
        if filename.endswith(f"{exp_id}.xlsx") and filename not in exclude_files:
            # Parse model, beams, max_new_tokens, temperature, and strategy from the filename
            parts = filename.replace(".xlsx", "").split('_')
            model = parts[0]
            dataset_name = parts[1]
            taxonomy_type = parts[2]
            beam_search = parts[3]  # Either 'beam' or 'sample'
            max_new_tokens = int(parts[4])
            temperature = float(parts[6])
            strategy = '_'.join(parts[8:]).replace("raw_table", "").replace("scores", "").replace(exp_id, "").replace("___", "_").strip('_')

            # Load the file into a DataFrame
            file_path = os.path.join(exp_dir, filename)
            df = pd.read_excel(file_path)

            # Determine if this is a raw table or a scores table
            if 'raw_table' in filename:
                # Add extra columns for model, beam_search, etc.
                df['Model'] = model
                df['Dataset Name'] = dataset_name
                df['Taxonomy Type'] = taxonomy_type
                df['Beam Search'] = beam_search
                df['Max New Tokens'] = max_new_tokens
                df['Temperature'] = temperature
                df['Strategy'] = strategy

                # Append to the unified raw table
                unified_raw_table = pd.concat([unified_raw_table, df], ignore_index=True)

            elif 'scores' in filename:
                # Add extra columns for model, beam_search, etc.
                df['Model'] = model
                df['Dataset Name'] = dataset_name
                df['Taxonomy Type'] = taxonomy_type
                df['Beam Search'] = beam_search
                df['Max New Tokens'] = max_new_tokens
                df['Temperature'] = temperature
                df['Strategy'] = strategy

                # Append to the unified scores table
                unified_scores_table = pd.concat([unified_scores_table, df], ignore_index=True)

    # Save the unified tables to Excel files
    desired_raw_table_column_order = [
        'Model', 'Dataset Name', 'Taxonomy Type', 'Strategy', 'Beam Search', 'Max New Tokens', 
        'Temperature', 'Query ID', 'Query', 'Answer', 'Run', 'Generated Response', 'Cleaned',
        'Is Correct', 'P', 'P_adj', 'Entropy', 'P_sem', 'Token Probabilities'
    ]
    desired_scores_column_order = [
        'Model', 'Dataset Name', 'Taxonomy Type', 'Strategy', 'Beam Search', 'Max New Tokens', 'Temperature',
        'Accuracy', 'AUCROC_sample', 'AUCPR_sample', 'AUCROC_adj', 'AUCPR_adj',
        'AUCROC_entr', 'AUCPR_entr', 'AUCROC_sem', 'AUCPR_sem'
    ]
    unified_raw_table = unified_raw_table[desired_raw_table_column_order]
    unified_scores_table = unified_scores_table[desired_scores_column_order]
    unified_raw_table.to_excel(unified_raw_table_path, index=False)
    unified_scores_table.to_excel(unified_scores_table_path, index=False)
    print(unified_raw_table.columns)
    print(unified_scores_table.columns)

    print(f"Unified raw table saved to {unified_raw_table_path}")
    print(f"Unified scores table saved to {unified_scores_table_path}")

# Example usage:
unify_excel_files("results/reliability_eval")

## 4. Find Duplicates among FKTC rows

In [ ]:
import os
import json
from collections import defaultdict

DATA_DIR = "/nfs/students/daro/data/MONITOR/FKTC"
DATA_FILES = [
    "P101", "P103", "P108", "P127", "P1376", "P1412", "P159", "P17",
    "P176", "P178", "P19", "P20", "P264", "P27", "P276", "P30", 
    "P364", "P37", "P495", "P740"
]

DATA_FILES_JSON = [f"{file_name}-subclass.json" for file_name in DATA_FILES]

def find_duplicates(data_dir, data_files_json):
    """
    Find duplicate rows in JSON files where duplicates are defined as having the same "subject".

    Parameters:
        data_dir (str): Directory where the dataset files are stored.
        data_files_json (list): List of dataset JSON file names to check for duplicates.

    Returns:
        None
    """
    for json_file in data_files_json:
        file_path = os.path.join(data_dir, json_file)
        subject_counts = defaultdict(int)

        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf8') as f:
                lines = [json.loads(line) for line in f.readlines()[1:]]  # Skip the first line with "relations"

                for entry in lines:
                    subject = entry['subject']
                    subject_counts[subject] += 1

            duplicates = [subject for subject, count in subject_counts.items() if count > 1]

            if duplicates:
                print(f"Duplicates found in file: {json_file}")
                for subject in duplicates:
                    print(f"  Duplicate subject: {subject}")
        else:
            print(f"File not found: {file_path}")

# Run the function to find duplicates
find_duplicates(DATA_DIR, DATA_FILES_JSON)


## 5. Plot Results

In [ ]:
import os
import pandas as pd
import plotly.express as px
from fpdf import FPDF

# Load the Excel file
file_path = "results/reliability_eval/unified_scores_table_08-24-4.xlsx"
df = pd.read_excel(file_path)

# Create a directory for saving plots
plots_dir = "plots/strategies_eval"
os.makedirs(plots_dir, exist_ok=True)

# Function to create and save plotly figures
def create_and_save_figures(df, plots_dir):
    # Plot 1: Accuracy vs. Strategy for each dataset (simplified without color)
    fig1 = px.box(
        df,
        x='Strategy',
        y='Accuracy',
        title='Accuracy by Strategy and Dataset',
        labels={'Strategy': 'Strategy', 'Accuracy': 'Accuracy'},
        points='all'
    )
    plot_file_1 = os.path.join(plots_dir, "plot_1_accuracy_by_strategy.png")
    fig1.write_image(plot_file_1)

    # Plot 2: Top 5 Strategies (Accuracy) across Datasets (simplified without color)
    top_5_strategies = df.groupby('Strategy')['Accuracy'].mean().nlargest(5).index
    fig2 = px.box(
        df[df['Strategy'].isin(top_5_strategies)],
        x='Strategy',
        y='Accuracy',
        title='Top 5 Strategies (Accuracy) by Dataset',
        labels={'Strategy': 'Strategy', 'Accuracy': 'Accuracy'},
        points='all'
    )
    plot_file_2 = os.path.join(plots_dir, "plot_2_top_5_strategies_accuracy.png")
    fig2.write_image(plot_file_2)

    # Plot 3: Accuracy vs. Dataset (this remains unchanged)
    fig3 = px.box(
        df,
        x='Dataset Name',
        y='Accuracy',
        title='Accuracy by Dataset',
        labels={'Dataset Name': 'Dataset Name', 'Accuracy': 'Accuracy'}
    )
    plot_file_3 = os.path.join(plots_dir, "plot_3_accuracy_by_dataset.png")
    fig3.write_image(plot_file_3)

    # Plot 4: Accuracy vs. Max New Tokens (simplified without color)
    fig4 = px.box(
        df,
        x='Max New Tokens',
        y='Accuracy',
        title='Accuracy by Max New Tokens',
        labels={'Max New Tokens': 'Max New Tokens', 'Accuracy': 'Accuracy'}
    )
    plot_file_4 = os.path.join(plots_dir, "plot_4_accuracy_by_max_new_tokens.png")
    fig4.write_image(plot_file_4)

    # Plot 5: AUCPR_sem vs. Strategy for each dataset (normal scale, restricted y-axis range)
    fig5 = px.box(
        df,
        x='Strategy',
        y='AUCPR_sem',
        title='AUCPR_sem by Strategy and Dataset (0.9 to 1.0)',
        labels={'Strategy': 'Strategy', 'AUCPR_sem': 'AUCPR_sem'},
        points='all'
    )
    fig5.update_yaxes(range=[0.9, 1.0])  # Restricted y-axis range
    plot_file_5 = os.path.join(plots_dir, "plot_5_aucpr_sem_by_strategy_restricted.png")
    fig5.write_image(plot_file_5)

    # Plot 6: Top 5 Strategies (AUCPR_sem) across Datasets (normal scale, restricted y-axis range)
    fig6 = px.box(
        df[df['Strategy'].isin(top_5_strategies)],
        x='Strategy',
        y='AUCPR_sem',
        title='Top 5 Strategies (AUCPR_sem) by Dataset (0.9 to 1.0)',
        labels={'Strategy': 'Strategy', 'AUCPR_sem': 'AUCPR_sem'},
        points='all'
    )
    fig6.update_yaxes(range=[0.9, 1.0])  # Restricted y-axis range
    plot_file_6 = os.path.join(plots_dir, "plot_6_top_5_strategies_aucpr_sem_restricted.png")
    fig6.write_image(plot_file_6)

    # Plot 7: AUCPR_sem vs. Dataset (normal scale, restricted y-axis range)
    fig7 = px.box(
        df,
        x='Dataset Name',
        y='AUCPR_sem',
        title='AUCPR_sem by Dataset (0.9 to 1.0)',
        labels={'Dataset Name': 'Dataset Name', 'AUCPR_sem': 'AUCPR_sem'}
    )
    fig7.update_yaxes(range=[0.9, 1.0])  # Restricted y-axis range
    plot_file_7 = os.path.join(plots_dir, "plot_7_aucpr_sem_by_dataset_restricted.png")
    fig7.write_image(plot_file_7)

    # Plot 8: AUCPR_sem vs. Max New Tokens (normal scale, restricted y-axis range)
    fig8 = px.box(
        df,
        x='Max New Tokens',
        y='AUCPR_sem',
        title='AUCPR_sem by Max New Tokens (0.9 to 1.0)',
        labels={'Max New Tokens': 'Max New Tokens', 'AUCPR_sem': 'AUCPR_sem'}
    )
    fig8.update_yaxes(range=[0.9, 1.0])  # Restricted y-axis range
    plot_file_8 = os.path.join(plots_dir, "plot_8_aucpr_sem_by_max_new_tokens_restricted.png")
    fig8.write_image(plot_file_8)

    return [
        (plot_file_1, "Accuracy by Strategy and Dataset"),
        (plot_file_2, "Top 5 Strategies (Accuracy) by Dataset"),
        (plot_file_3, "Accuracy by Dataset"),
        (plot_file_4, "Accuracy by Max New Tokens"),
        (plot_file_5, "AUCPR_sem by Strategy and Dataset (0.9 to 1.0)"),
        (plot_file_6, "Top 5 Strategies (AUCPR_sem) by Dataset (0.9 to 1.0)"),
        (plot_file_7, "AUCPR_sem by Dataset (0.9 to 1.0)"),
        (plot_file_8, "AUCPR_sem by Max New Tokens (0.9 to 1.0)")
    ]

# Generate and save the figures
plots = create_and_save_figures(df, plots_dir)

# Create a PDF document
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)

# Add plots and descriptions to the PDF
for plot_file, desc in plots:
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.multi_cell(0, 10, desc)
    pdf.ln(10)
    pdf.image(plot_file, w=pdf.w - 30)

# Save the PDF
pdf_path = os.path.join(plots_dir, "strategies_evaluation_plots.pdf")
pdf.output(pdf_path, "F")

print(f"PDF generated and saved as '{pdf_path}'")

In [ ]:
from fpdf import FPDF

# Function to get prompt
def get_prompt(query, strategy):
    if strategy == "Fact Statement":
        return f"{query} Fact:"
    elif strategy == "Completion":
        return f"{query} The answer is:"
    elif strategy == "Definitive Statement":
        return f"The answer to the question '{query}' is:"
    elif strategy == "Fill-in-the-Blank":
        return f"{query} The answer is: _____.\nAnswer:"
    elif strategy == "Structured Answer Prompt":
        return f"Question: {query}\nAnswer (one word):"
    elif strategy == "Direct Instruction":
        return f"Please answer the following question in one word.\nQuestion: {query}\nAnswer:"
    elif strategy == "Contextual Prompts":
        return f"{query} (Please answer in one word)"
    elif strategy == "Question-Answer Pairs":
        return f"QSTN: What is the capital of France?\nANSR: Paris\nQSTN: What is the capital of Germany?\nANSR: Berlin\nQSTN: {query}\nANSR:"
    elif strategy == "Direct Answer":
        return f"Please provide a short, direct answer to the following question: {query} Answer:"
    elif strategy == "Q&A Format":
        return f"Q: {query}\nA:"
    elif strategy == "Instructional":
        return f"Answer the following question in one or two words: {query}"
    elif strategy == "Summary":
        return f"Summarize the answer to the following question: {query}"
    elif strategy == "Echo":
        return f"{query} {query}"
    elif strategy == "True Completion":
        return f"{query} The true answer is:"
    elif strategy == "Direct Completion":
        return f"{query} Answer:"
    elif strategy == "Answer Completion":
        return f"{query} The correct answer is:"
    elif strategy == "Direct Query":
        return f"{query}?"
    elif strategy == "Factual Retrieval":
        return f"Based on known facts, what is the answer to the following: {query}?"
    elif strategy == "First Thought":
        return f"What is the first thing that comes to your mind when asked: {query}?"
    elif strategy == "Deductive Reasoning":
        return f"Given these facts: 1) Paris is the capital of France. 2) The Eiffel Tower is located in Paris. 3) French is the official language of France. Deduce the answer to the following: {query}."
    else:
        return query

# Define the base query
base_query = "Which language is mainly spoken in Japan?"

# Define all the strategies
strategies = [
    "Fact Statement", "Completion", "Definitive Statement", "Fill-in-the-Blank", 
    "Structured Answer Prompt", "Direct Instruction", "Contextual Prompts", 
    "Question-Answer Pairs", "Direct Answer", "Q&A Format", "Instructional", 
    "Summary", "Echo", "True Completion", "Direct Completion", "Answer Completion",
    "Direct Query", "Factual Retrieval", "First Thought", "Deductive Reasoning"
]

# Collect strategy prompts
strategy_prompts = [(strategy, get_prompt(base_query, strategy)) for strategy in strategies]

# Dataset information
datasets = {
    "toy-qa-dataset": {
        "relations": ["Multiple question types: places, adjectives, persons, abstract answers, numerical answers, languages, companies."],
        "example": {"subject": "How would you describe the taste of a lemon?", "object": "Sour"},
        "type": "mixed"
    },
    "P17": {
        "relations": ["Where is [X] located?"],
        "example": {"subject": "Eibenstock", "object": "Germany"},
        "type": "location"
    },
    "P19": {
        "relations": ["Where was [X] born?"],
        "example": {"subject": "Allan Peiper", "object": "Alexandra"},
        "type": "person-birthplace"
    },
    "P20": {
        "relations": ["In what place did [X] pass away?"],
        "example": {"subject": "Akihiko Saito", "object": "Iraq"},
        "type": "person-death-place"
    },
    "P27": {
        "relations": ["What country is [X] a citizen of?"],
        "example": {"subject": "Rubens Barrichello", "object": "Brazil"},
        "type": "person-citizenship"
    },
    "P30": {
        "relations": ["Which continent is [X] located in?"],
        "example": {"subject": "Lavoisier Island", "object": "Antarctica"},
        "type": "location-continent"
    },
    "P37": {
        "relations": ["What language is the official language of [X]?"],
        "example": {"subject": "Azad Kashmir", "object": "Urdu"},
        "type": "location-language"
    },
    "P101": {
        "relations": ["What is [X]'s area of expertise?"],
        "example": {"subject": "Alan Turing", "object": "logic"},
        "type": "person-expertise"
    },
    "P103": {
        "relations": ["What is the native language of [X]?"],
        "example": {"subject": "Louis Jules Trochu", "object": "French"},
        "type": "person-language"
    },
    "P108": {
        "relations": ["Which organization does [X] work for?"],
        "example": {"subject": "Steve Jobs", "object": "Apple"},
        "type": "person-employer"
    },
    "P127": {
        "relations": ["Which company is the owner of [X]?"],
        "example": {"subject": "iWork", "object": "Apple"},
        "type": "company-ownership"
    },
    "P159": {
        "relations": ["In what city is [X] headquartered?"],
        "example": {"subject": "Chandos Records", "object": "Colchester"},
        "type": "company-headquarters"
    },
    "P176": {
        "relations": ["What is the manufacturer of [X]?"],
        "example": {"subject": "Fiat Multipla", "object": "Fiat"},
        "type": "product-manufacturer"
    },
    "P178": {
        "relations": ["Which company is the creator of [X]?"],
        "example": {"subject": "MessagePad", "object": "Apple"},
        "type": "product-creator"
    },
    "P264": {
        "relations": ["What is the record label for [X]?"],
        "example": {"subject": "Buddy Holly", "object": "Decca"},
        "type": "music-record-label"
    },
    "P276": {
        "relations": ["What is the location of [X]?"],
        "example": {"subject": "Eiffel Tower", "object": "Paris"},
        "type": "general-location"
    },
    "P364": {
        "relations": ["What is the native language of [X]?"],
        "example": {"subject": "The Scarlet Flower", "object": "Russian"},
        "type": "media-language"
    },
    "P495": {
        "relations": ["Which country was [X] created in?"],
        "example": {"subject": "Soppressata", "object": "Italy"},
        "type": "origin-country"
    },
    "P740": {
        "relations": ["Which city or region or country was [X] founded in?"],
        "example": {"subject": "Pianos Become the Teeth", "object": "Baltimore"},
        "type": "organization-foundation-location"
    },
    "P1376": {
        "relations": ["Which country's or administrative division's capital is [X]?"],
        "example": {"subject": "Edmonton", "object": "Alberta"},
        "type": "capital-location"
    },
    "P1412": {
        "relations": ["What language did [X] previously speak to communicate?"],
        "example": {"subject": "Iginio Ugo Tarchetti", "object": "Italian"},
        "type": "person-previous-language"
    }
}

# Create a PDF
pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", "B", 16)

# Add base query
pdf.cell(200, 10, "Base Query:", ln=True, align='C')
pdf.set_font("Arial", "", 14)
pdf.multi_cell(0, 10, base_query)
pdf.ln(10)

# Add strategies and prompts
pdf.set_font("Arial", "B", 16)
pdf.cell(200, 10, "Prompting Strategies:", ln=True, align='C')
pdf.set_font("Arial", "", 12)
for strategy, prompt in strategy_prompts:
    pdf.set_font("Arial", "B", 14)
    pdf.multi_cell(0, 10, f"Strategy: {strategy}")
    pdf.set_font("Arial", "", 12)
    pdf.multi_cell(0, 10, f"Prompt: {prompt}")
    pdf.ln(5)

# Add dataset descriptions
pdf.set_font("Arial", "B", 16)
pdf.cell(200, 10, "Datasets Description:", ln=True, align='C')
pdf.set_font("Arial", "", 12)
for dataset_name, details in datasets.items():
    pdf.set_font("Arial", "B", 14)
    pdf.multi_cell(0, 10, f"Dataset: {dataset_name}")
    pdf.set_font("Arial", "", 12)
    pdf.multi_cell(0, 10, f"Type/Category: {details['type']}")
    pdf.multi_cell(0, 10, f"First Relation: {details['relations'][0]}")
    example = details['example']
    pdf.multi_cell(0, 10, f"Example: {example['subject']} - {example['object']}")
    pdf.ln(5)

# Save the PDF to file
pdf.output("results/reliability_eval/FKTC_data_and_strategies.pdf")